# ColorizeGAN v2 - training on Google Colab
Ye notebook v2 model train karta hai (Lab colour space, U-Net + PatchGAN, 256x256). Har cell ke upar uska simple explanation hai. Cells ko upar se neeche ek-ek karke run karo.

## 1. GPU check
Menu: *Runtime > Change runtime type > T4 GPU*. Bina GPU ke training bahut slow hogi.

In [ ]:
# Colab ka naya TensorFlow (2.16+) Keras 3 use karta hai; ye code Keras 2 (tf_keras) ke liye hai.
# Ye cell sabse pehle chalao (runtime restart ke baad bhi).
import os, subprocess, sys
os.environ['TF_USE_LEGACY_KERAS'] = '1'
tf_version = subprocess.check_output(
    [sys.executable, '-c', 'import tensorflow as t; print(t.__version__)'], text=True).strip()
major_minor = '.'.join(tf_version.split('.')[:2])
print('TensorFlow', tf_version)
!pip install -q "tf_keras~={major_minor}.0"


In [ ]:
import tensorflow as tf
print('TF', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
!nvidia-smi | head -n 15

## 2. Google Drive mount
Trained model aur checkpoints Drive me save honge, taaki Colab disconnect hone par bhi na jaye.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/ColorizeGAN/v2'
!mkdir -p "$OUT_DIR"

## 3. Repo clone aur packages
Colab me TensorFlow pehle se hota hai - usse reinstall mat karo. Model `.h5` (Keras 2 format) me save hota hai. Agar Colab ka TensorFlow 2.16+ (Keras 3) ho aur `.h5` save/load me error aaye, to pehle `!pip install tf_keras` aur cell ke upar `import os; os.environ['TF_USE_LEGACY_KERAS']='1'` chalao (restart ke baad), ya Colab me TF 2.15 install karo.

In [ ]:
!git clone https://github.com/kripashankarcs3/ColorizeGAN.git /content/ColorizeGAN
%cd /content/ColorizeGAN
# Jis branch me v2 code hai (merge ke baad 'main' hi chalega):
# !git checkout feat/model-v2
!pip install -q datasets tqdm pillow scipy scikit-learn
# Optional metric (LPIPS):
!pip install -q lpips

## 4. Dataset download
`download_dataset.py` COCO ke chhote subset ko `dataset/train/color` aur `dataset/validation/color` me save karta hai. Default sirf 2000 train images hain - better result ke liye `TRAIN_LIMIT` badhao (neeche 20000 kiya hai).

In [ ]:
!sed -i 's/TRAIN_LIMIT = 2000/TRAIN_LIMIT = 20000/' download_dataset.py
!python download_dataset.py

## 5. Training
Generator loss = 100 x L1(ab) + adversarial. `--mixed-precision` GPU par fast hai. Pehle `--max-steps 20` se chhota test kar sakte ho. Checkpoints aur sample images Drive me `OUT_DIR` me aate hain, final model `CCGAN_v2.h5`.

In [ ]:
!python -m app.train_v2 --data-dir dataset --epochs 20 --batch-size 16 --image-size 256 \
    --out-dir "$OUT_DIR" --mixed-precision --sample-every 500

## 6. Sample results dekho
Har row: grayscale | model ki colorization | asli photo.

In [ ]:
import glob
from IPython.display import Image, display
samples = sorted(glob.glob(f'{OUT_DIR}/samples/*.png'))
print(samples[-3:])
if samples:
    display(Image(samples[-1]))

## 7. Evaluation (Phase 6)
PSNR / SSIM / LPIPS validation images par, v1 aur v2 ke liye. Table Drive me `results.md` me save hoti hai.

In [ ]:
!python -m tools.evaluate --data-dir dataset --limit 200 --v2-model "$OUT_DIR/CCGAN_v2.h5" --out "$OUT_DIR/results.md"

## 8. Agla step
`CCGAN_v2.h5` ko Drive se download karo, ONNX me export karo (`tools/export_onnx.py`, fp16 se size aadha ho sakta hai), aur website me use karo.